In [1]:
# Step 1: Import necessary libraries

import pandas as pd

from datetime import datetime, timedelta
import xlsxwriter

import pandas as pd

from datetime import timedelta
from plotly.subplots import make_subplots
import plotly.io as pio  # Imported 'pio'
from datetime import datetime, timedelta
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import xlsxwriter

import plotly.graph_objs as go
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import pandas as pd
from datetime import datetime, timedelta
import pandas as pd
# Assuming you've already imported the necessary libraries like blp.
import pandas as pd
import win32com.client as win32
import time
import win32com.client
from win32com.client import makepy
import win32com.client as win32
import os
import pandas as pd
import win32com.client as win32
import time
import pandas as pd
import re
import psutil
import os
import win32com.client as win32
import time
from datetime import datetime, timedelta
import win32com.client as win32
import math
import win32com.client as win32
from win32com.client import constants
import pandas as pd
import os
from win32com import client
from docx import Document
from docx.shared import Pt, RGBColor, Inches
from docx.enum.text import WD_PARAGRAPH_ALIGNMENT
from docx.enum.table import WD_ALIGN_VERTICAL
from docx.oxml import OxmlElement
from docx.oxml.ns import qn

The create_table function now configures a table with four rows on each page. Two rows are designated for titles ("Title" and "Fundamental / Technical Idea") with blue background shading and white text. The other two rows are for the content, specifically designated for "Chart placeholder" and "Fundamental / Technical Idea" as per your description.
The script ensures that each page follows the same format, with the specified titles and content in their designated places.
This solution extends the document to include six pages, each with the described configuration, maintaining the established aesthetic and formatting preferences you've specified.

In [2]:
from docx import Document
from docx.shared import Inches, Pt, RGBColor
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.enum.text import WD_PARAGRAPH_ALIGNMENT
from docx.enum.table import WD_ALIGN_VERTICAL
import os

def close_and_save_if_open(file_path):
    try:
        word = client.Dispatch("Word.Application")
        for doc in word.Documents:
            if doc.FullName.lower() == file_path.lower():
                doc.Save()
                doc.Close()
                break
    except Exception as e:
        print(f"An error occurred while closing the document: {e}")

def set_cell_margins(cell, top, bottom, left, right):
    properties = cell._element.get_or_add_tcPr()
    margins = OxmlElement('w:tcMar')
    for name, value in {"top": top, "bottom": bottom, "left": left, "right": right}.items():
        margin = OxmlElement(f'w:{name}')
        margin.set(qn('w:w'), str(value))
        margin.set(qn('w:type'), 'dxa')
        margins.append(margin)
    properties.append(margins)

def remove_space_before_after_for_all_paragraphs(doc):
    for paragraph in doc.paragraphs:
        paragraph.paragraph_format.space_before = Pt(0)
        paragraph.paragraph_format.space_after = Pt(0)
    for table in doc.tables:
        for row in table.rows:
            for cell in row.cells:
                for paragraph in cell.paragraphs:
                    paragraph.paragraph_format.space_before = Pt(0)
                    paragraph.paragraph_format.space_after = Pt(0)

def create_document_content(doc):
    # Adjust margins for a narrow layout
    section = doc.sections[-1]  # Get the last section
    section.top_margin = Inches(0.5)
    section.bottom_margin = Inches(0.5)
    section.left_margin = Inches(0.5)
    section.right_margin = Inches(0.5)

    desired_font_color = RGBColor(0x00, 0x21, 0x40)

    style = doc.styles['Normal']
    font = style.font
    font.name = 'Arial(Body)'
    font.size = Pt(10)
    font.color.rgb = desired_font_color

    title = doc.add_heading(level=1)
    run = title.add_run('Template')
    run.font.size = Pt(20)
    run.font.color.rgb = desired_font_color
    title.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

    border_paragraph = doc.add_paragraph()
    full_width_line = '─' * 95
    border_run = border_paragraph.add_run(full_width_line)
    border_run.font.size = Pt(8)
    border_run.font.color.rgb = desired_font_color
    border_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

    for i in range(3):
        p = doc.add_paragraph(style='List Bullet')
        run = p.add_run(f'Bullet point {i + 1}')
        run.font.color.rgb = desired_font_color
        p.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY

    remove_space_before_after_for_all_paragraphs(doc)

    create_table(doc)  # Call to create a table with 2 chart placeholders

    border_paragraph = doc.add_paragraph()
    border_run = border_paragraph.add_run(full_width_line)
    border_run.font.size = Pt(8)
    border_run.font.color.rgb = desired_font_color
    border_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

    remove_space_before_after_for_all_paragraphs(doc)

def create_table(doc):
    # Adjusted to create 2 rows, one for titles and one for contents
    table = doc.add_table(rows=2, cols=3)
    table.autofit = False
    table.allow_autofit = False
    wide_col_width = Inches(3.6)
    narrow_col_width = Inches(0.2)

    for row in table.rows:
        row.cells[0].width = wide_col_width
        row.cells[1].width = narrow_col_width
        row.cells[2].width = wide_col_width
        for cell in row.cells:
            set_cell_margins(cell, 0, 0, 50, 50)

    # Title row with blue background and white text
    for j, cell in enumerate(table.rows[0].cells):
        cell.vertical_alignment = WD_ALIGN_VERTICAL.CENTER
        if j != 1:  # Skip the middle narrow column
            shading_elm = OxmlElement("w:shd")
            shading_elm.set(qn("w:fill"), "002140")
            cell._tc.get_or_add_tcPr().append(shading_elm)
            paragraph = cell.paragraphs[0]
            run = paragraph.add_run('Title')
            run.font.size = Pt(10)
            run.font.color.rgb = RGBColor(255, 255, 255)
            run.font.bold = True
            paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

    # Content row with specific placeholders
    contents = ["Fundamental / Technical Idea", "Chart placeholder"]
    for i, cell in enumerate(table.rows[1].cells):
        if i != 1:  # Apply for the wide columns only
            paragraph = cell.paragraphs[0]
            run = paragraph.add_run(contents[i//2])  # i//2 will be 0 for first and 1 for third cell
            run.font.size = Pt(10)
            run.font.color.rgb = RGBColor(0x00, 0x21, 0x40)
            paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

def create_document_template():
    file_path = 'template.docx'
    abs_file_path = os.path.abspath(file_path)

    close_and_save_if_open(abs_file_path)

    doc = Document()

    for _ in range(6):  # Six pages of content
        create_document_content(doc)
        if _ < 5:  # Add a page break after each section except the last
            doc.add_page_break()

    doc.save(file_path)

    if os.name == 'nt':
        os.startfile(file_path)
    else:
        os.system(f'open "{file_path}"' if os.name == 'posix' else f'xdg-open "{file_path}"')

create_document_template()
